# DecoupleNet LoveDA Egitimi — Google Colab

**Drive merkezli:** Veri seti ve modeller Google Drive'da kalici olarak saklanir.
- **Ilk calisma:** Veri setini indirir (~10 dk), Drive'a kaydeder
- **Sonraki calismalar:** Drive'dan okur, sadece egitimi baslatir (~2 dk)

## Hucresler
1. Drive bagla + kurulum
2. Veri setini hazirla (sadece ilk kez indirir)
3. Egitimi baslat
4. TensorBoard

In [ ]:
# ===================== HUCRE 1: DRIVE BAGLA + KURULUM =====================
import os, sys

# Google Drive bagla
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print('Drive baglandi!')
else:
    print('Drive zaten bagli.')

# Sabit yollar
DRIVE_BASE = '/content/drive/MyDrive/DecoupleNet'
DATA_ROOT = f'{DRIVE_BASE}/data/LoveDA'
WEIGHTS_DIR = f'{DRIVE_BASE}/backbone_weights'
CKPT_DIR = f'{DRIVE_BASE}/model_weights/loveda'

# Drive'da gerekli dizinleri olustur
for d in [DATA_ROOT, WEIGHTS_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# GPU kontrolu
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB)')
else:
    print('UYARI: GPU yok! Runtime -> Change runtime type -> T4 GPU')

# Repo klonla
if not os.path.exists('/content/DecoupleNet'):
    !git clone https://github.com/myrisee/DecoupleNet.git /content/DecoupleNet
else:
    !cd /content/DecoupleNet && git pull origin main

# Backbona agirliklarini Drive'a indir (yoksa)
weights_path = f'{WEIGHTS_DIR}/DecoupleNet_D2.pth'
if not os.path.exists(weights_path):
    print('Backbone agirlik indiriliyor (126 MB)...')
    !wget -q --show-progress -O "{weights_path}" https://github.com/lwCVer/DecoupleNet/releases/download/weights/DecoupleNet_D2.pth
    print('Indirildi!')
else:
    print(f'Backbone agirlik mevcut: {os.path.getsize(weights_path)/1024**2:.0f} MB')

# Repo'daki backbone_weights dizinine symlink (model ../backbone_weights/ ile okuyor)
repo_bw = '/content/DecoupleNet/segmentation/backbone_weights'
if os.path.islink(repo_bw):
    os.unlink(repo_bw)
elif os.path.exists(repo_bw):
    !rm -rf "{repo_bw}"
os.symlink(WEIGHTS_DIR, repo_bw)

# Bagimliliklari kur
print('Bagimliliklar kuruluyor...')
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q timm pytorch-lightning==1.9.0
!pip install -q albumentations einops ttach pytorch-toolbelt
!pip install -q opencv-python-headless scipy matplotlib tqdm addict antialiased-cnns huggingface_hub

%cd /content/DecoupleNet/segmentation
print(f'\nHazir! Calisma dizini: {os.getcwd()}')

In [ ]:
# ===================== HUCRE 2: VERI SETINI HAZIRLA =====================
# Drive'da yoksa indirir (ilk kez ~10 dk), varsa atlar (1 saniye)

import os, glob

train_ready = os.path.exists(f'{DATA_ROOT}/Train/masks_png_convert')
val_ready = os.path.exists(f'{DATA_ROOT}/Val/masks_png_convert')

if train_ready and val_ready:
    ti = len(glob.glob(f'{DATA_ROOT}/Train/images_png/*.png'))
    vi = len(glob.glob(f'{DATA_ROOT}/Val/images_png/*.png'))
    print(f'Veri seti Drive\'da hazir! Train: {ti}, Val: {vi}')
else:
    print('LoveDA veri setini HuggingFace\'den indiriyoruz (ilk kez)...')
    print('Bu islem ~10 dakika surecek. Sonraki calismalarda atlaniyor.')
    !python tools/prepare_loveda_colab.py --data-root "{DATA_ROOT}"
print('\nHazir!')

In [ ]:
# ===================== HUCRE 3: EGITIMI BASLAT =====================
# Tahmini: T4 ile ~12-15 saat (30 epoch)
# Colab ucretsiz hesap: 12 saat zaman asimi
# Checkpoint'ler Drive'a kaydedilir -> session bitince devam edebilirsiniz

%cd /content/DecoupleNet/segmentation

!python train_supervision.py -c config/loveda/train_decouplenet_colab.py

In [ ]:
# ===================== HUCRE 4: TENSORBOARD =====================
# Egitim sirasinda calistirarak metrikleri izleyin

%load_ext tensorboard
%tensorboard --logdir /content/DecoupleNet/segmentation/lightning_logs